# Register the v1 RFC forecast dataset and copy its legacy history

Sets `public` up for `v1-ingest-nwps-rfc-streamflow` (`prefect-workflows/v1/flows/ingest_nwps.py`):

| v1 (`public`) | what |
| --- | --- |
| `nwpsrfc_streamflow_forecast` | the RFC streamflow forecasts from the NWPS API, every step (1, 3, 6 hours): legacy's configuration of that name, whose two variables (`streamflow_hourly_inst`, `streamflow_6hr_inst`) become one dataset |
| `nwpsrfc_streamflow_forecast_series` | its series (`nwpsrfc-<lid>`, legacy's ids), each with its step as `resolution` (`PT1H`, `PT3H`, `PT6H`) |
| `usgs_nwpsrfc` (crosswalk) | the USGS gages to their RFC gauges (legacy's crosswalk): pairs `usgs_inst_streamflow` with the forecasts, and is where the ingest finds its gauges |

Legacy's reference times are already one step before each forecast's first value, as the ingest
sets them (issue #258); legacy has no 3-hour forecasts (the new ingest adds them).

```yaml
inputs:
  primary: {dataset: public.usgs_inst_streamflow}
  secondary: [{dataset: public.nwpsrfc_streamflow_forecast}]
pair: {relation: public.usgs_nwpsrfc}
```

**Run it as a platform admin, on use1, before the ingest's deployment** (its hourly schedule
needs the dataset and the crosswalk). Spark writes the rows straight into the `public` Iceberg
tables (as `06_nwm_points`). **Re-runnable:** registering skips what exists; the crosswalk's
links are written only while it's empty; series are added, never changed; each month copied
replaces that month. Start with **2. Count**: it writes nothing.

In [ ]:
import os

import pandas as pd
import requests
import teehr
from teehr.evaluation.spark_session_utils import create_spark_session

teehr.__version__

## 1. Settings

In [ ]:
LEGACY_CONFIGURATION = "nwpsrfc_streamflow_forecast"
DATASET = "nwpsrfc_streamflow_forecast"
SERIES = f"{DATASET}_series"  # made by registering the dataset
CROSSWALK, OBSERVED = "usgs_nwpsrfc", "usgs_inst_streamflow"
#: Legacy's variables -> the step they were (the series' resolution).
STEPS = {"streamflow_hourly_inst": "PT1H", "streamflow_6hr_inst": "PT6H"}
MAX_LEAD = "45D"  # longer than any RFC forecast (about a month)
SOURCE = ("NWS National Water Prediction Service (NWPS) API, RFC stage/flow forecasts; history "
          "before the v1 ingest copied from the legacy TEEHR warehouse")

CONTROL_PLANE = os.environ.get(
    "CONTROL_PLANE_URL", "http://teehr-control-plane.teehr-hub.svc.cluster.local:8000"
)
VERIFY_TLS = "local.app.garden" not in CONTROL_PLANE  # KinD's certificate is self-signed
LEGACY, PUBLIC = "iceberg.teehr", "iceberg.public"

In [ ]:
# Local mode: 27 M rows since 2026-02.
spark = create_spark_session()
spark.conf.set("spark.sql.iceberg.check-nullability", "false")
spark.conf.set("spark.sql.session.timeZone", "UTC")  # months are UTC, as Iceberg's partitions


def access_token():
    """Your access token, refreshed when it's about to expire (JupyterHub sets it)."""
    try:
        from teehr.utils.keycloak_auth import refresh_user_token_from_env
    except ImportError:  # a kernel without the helper: the token as JupyterHub set it
        return os.environ["POLARIS_USER_TOKEN"]
    access, _refresh = refresh_user_token_from_env()
    return access


api = requests.Session()
api.verify = VERIFY_TLS


def call(method, path, **kwargs):
    resp = api.request(method, f"{CONTROL_PLANE}/v1{path}",
                       headers={"Authorization": f"Bearer {access_token()}"}, **kwargs)
    if resp.status_code >= 400 and resp.status_code != 404:
        raise RuntimeError(f"{method} {path}: HTTP {resp.status_code}: {resp.text[:1000]}")
    return resp


def exists(name):
    return call("GET", f"/projects/public/datasets/{name}").status_code == 200


def register(body):
    if exists(body["name"]):
        print(f"public.{body['name']}: exists")
        return
    call("POST", "/projects/public/datasets", json=body)
    print(f"public.{body['name']}: registered")


def month_range(column, month, lead="0D"):
    """A time range filter for one UTC month (prunes the scan), widened by ``lead``."""
    start = pd.Period(month, freq="M")
    end = (start + 1).start_time + pd.Timedelta(lead)
    return (f"{column} >= TIMESTAMP '{start.start_time:%Y-%m-%d} 00:00:00' "
            f"AND {column} < TIMESTAMP '{end:%Y-%m-%d %H:%M:%S}'")


LEGACY_ROWS = (f"{LEGACY}.secondary_timeseries WHERE configuration_name = "
               f"'{LEGACY_CONFIGURATION}' AND value IS NOT NULL")

me = call("GET", "/projects/public").json()
print(f"public: {me['status']}; your role there: {me.get('my_role')}")

## 2. Count (writes nothing)

Legacy's rows per variable, and per month of `reference_time`; the lead of each forecast's first
value must be one step (a few forecasts lost their first values to no-data).

In [ ]:
%%time
spark.sql(f"""
    SELECT variable_name, unit_name, count(*) AS rows, count(DISTINCT location_id) AS gauges,
           min(reference_time) AS first, max(reference_time) AS last
    FROM {LEGACY_ROWS} GROUP BY 1, 2 ORDER BY 1
""").show(truncate=False)
spark.sql(f"""
    SELECT variable_name, timestampdiff(HOUR, reference_time, min(value_time)) AS first_lead_h,
           count(*) AS forecasts
    FROM {LEGACY_ROWS} GROUP BY variable_name, location_id, reference_time
""").groupBy("variable_name", "first_lead_h").count().orderBy("variable_name", "first_lead_h").show(8)
legacy = spark.sql(f"""
    SELECT date_format(reference_time, 'yyyy-MM') AS month, count(*) AS legacy
    FROM {LEGACY_ROWS} GROUP BY 1 ORDER BY 1
""").toPandas()
legacy

## 3. Register the dataset and the crosswalk (as you)

A forecast of instantaneous streamflow in m3/s (legacy's `m^3/s`: the counts above must show only
that unit), with no location layer: the crosswalk pairs its gauges with the USGS gages.

In [ ]:
register({
    "type": "timeseries", "name": DATASET, "role": "forecast",
    "variable": "streamflow", "statistic": "inst", "source": SOURCE,
    "description": ("RFC streamflow forecasts at gauges from the NWS National Water Prediction "
                    "Service (each gauge's latest; 1, 3 or 6 hour steps, as the series' "
                    "resolution); reference time one step before the first value"),
})
register({"type": "crosswalk", "name": CROSSWALK, "left": OBSERVED, "right": DATASET,
          "source": "Legacy TEEHR warehouse (location crosswalks)",
          "description": "USGS gages to their RFC (NWPS) forecast gauges"})

## 4. The crosswalk's links and the series

The links from legacy's crosswalk, only while the crosswalk is empty. Only gauges with legacy
forecasts are added to the series table, each with its latest forecast's step: the ingest adds
the others, with theirs, when they first forecast.

In [ ]:
%%time
if spark.table(f"{PUBLIC}.{CROSSWALK}").limit(1).count() == 0:
    spark.sql(f"""
        SELECT DISTINCT primary_location_id AS left_key, secondary_location_id AS right_key
        FROM {LEGACY}.location_crosswalks
        WHERE primary_location_id LIKE 'usgs-%' AND secondary_location_id LIKE 'nwpsrfc-%'
    """).writeTo(f"{PUBLIC}.{CROSSWALK}").append()
print(f"{CROSSWALK}: {spark.table(f'{PUBLIC}.{CROSSWALK}').count():,} links")

steps = " ".join(f"WHEN '{v}' THEN '{r}'" for v, r in STEPS.items())
spark.sql(f"""
    MERGE INTO {PUBLIC}.{SERIES} t
    USING (SELECT location_id AS series_key,
                  CASE max_by(variable_name, reference_time) {steps} END AS resolution
           FROM {LEGACY_ROWS} GROUP BY location_id) s
    ON t.series_key = s.series_key
    WHEN NOT MATCHED THEN INSERT (series_key, location_key, location_layer, period, resolution)
        VALUES (s.series_key, NULL, NULL, NULL, s.resolution)
""")
spark.table(f"{PUBLIC}.{SERIES}").groupBy("resolution").count().show()

## 5. Copy the history

A month of `reference_time` at a time, each replacing that month of the v1 table. Values are
legacy's float32, as doubles.

In [ ]:
START = None  # e.g. "2026-10" to resume from a month


def copy_month(month):
    """Replace one UTC month (of reference_time) of the v1 dataset with the legacy rows."""
    # Legacy is partitioned by month of value_time: bound it too, so the scan reads two months
    # of partitions, not all.
    spark.sql(f"""
        SELECT location_id AS series_key, reference_time, value_time,
               CAST(value AS DOUBLE) AS value
        FROM {LEGACY_ROWS}
          AND {month_range("reference_time", month)}
          AND {month_range("value_time", month, MAX_LEAD)}
    """).writeTo(f"{PUBLIC}.{DATASET}").overwritePartitions()

In [ ]:
%%time
for month in legacy.month:
    if START and month < START:
        continue
    copy_month(month)
    print(f"{month}: copied")

## 6. Check: v1 against legacy, month by month

In [ ]:
%%time
v1 = spark.sql(f"""
    SELECT date_format(reference_time, 'yyyy-MM') AS month, count(*) AS v1
    FROM {PUBLIC}.{DATASET} GROUP BY 1
""").toPandas()
both = legacy.merge(v1, on="month", how="outer").fillna(0)
both["matches"] = both.legacy == both.v1
print(f"{int(both.v1.sum()):,} rows; every month matches: {both.matches.all()}")
both

**After it:** merge the ingest's PR (its deployment's hourly schedule is on). Its first run
writes only forecasts newer than each gauge's latest copied one; legacy keeps ingesting until
the two match.